# 01 · Data prep
Loads the raw files in `data/raw/`, fixes team names, keeps the 19 competitive tournaments from 2000 onwards, and writes the cleaned match table.

**Reads:** `data/raw/results_new.csv`, `goalscorers.csv`, `shootouts.csv`, `former_names.csv`

**Writes:** `data/processed/clean_matches.csv`, `data/wc2026/upcoming_fixtures.csv`, and overwrites `data/raw/goalscorers.csv` and `shootouts.csv` with renamed teams (re-download them to start over).

See the README for where to download the raw files.

In [ ]:
import pandas as pd
import numpy as np
import os

for d in ['../data/processed', '../data/wc2026']:
    os.makedirs(d, exist_ok=True)

# ---- Load ----
results_new  = pd.read_csv('../data/raw/results_new.csv', parse_dates=['date'])
former_names = pd.read_csv('../data/raw/former_names.csv', parse_dates=['start_date','end_date'])
goals_df     = pd.read_csv('../data/raw/goalscorers.csv', parse_dates=['date'])
shootouts_df = pd.read_csv('../data/raw/shootouts.csv', parse_dates=['date'])

print(f"Raw results_new.csv : {results_new.shape}")

# ---- Step 1: Set aside unplayed fixtures (null scores) -- these are what we'll predict later ----
upcoming_fixtures = results_new[results_new['home_score'].isnull()].copy()
results_new = results_new[results_new['home_score'].notnull()].copy()
print(f"\nUpcoming (unplayed) fixtures set aside: {len(upcoming_fixtures)}")
print(upcoming_fixtures[['date','home_team','away_team','tournament']].to_string(index=False))

# ---- Step 2: Deduplicate ----
before = len(results_new)
results_new = results_new.drop_duplicates(
    subset=['date','home_team','away_team','home_score','away_score'], keep='first')
print(f"\nDropped {before - len(results_new)} exact duplicate row(s)")

dupe_keys = results_new.duplicated(subset=['date','home_team','away_team'], keep=False)
conflicting = results_new[dupe_keys]
if len(conflicting) > 0:
    print(f"Dropping {len(conflicting)} row(s) with conflicting scores for the same fixture (unverifiable):")
    print(conflicting[['date','home_team','away_team','home_score','away_score']].to_string(index=False))
    results_new = results_new[~dupe_keys]

# ---- Step 3: Former-name remapping, point-in-time correct, applied consistently everywhere ----
def build_remap_fn(df):
    def remap(row_date, team_name):
        m = df[(df['former'] == team_name) &
               ((df['start_date'].isnull()) | (df['start_date'] <= row_date)) &
               ((df['end_date'].isnull())   | (df['end_date']   >= row_date))]
        return m.iloc[0]['current'] if len(m) > 0 else team_name
    return remap

remap = build_remap_fn(former_names)

for col in ['home_team', 'away_team']:
    results_new[col] = results_new.apply(lambda r: remap(r['date'], r[col]), axis=1)
for col in ['home_team', 'away_team', 'team']:
    goals_df[col] = goals_df.apply(lambda r: remap(r['date'], r[col]), axis=1)
for col in ['home_team', 'away_team', 'winner', 'first_shooter']:
    shootouts_df[col] = shootouts_df.apply(lambda r: remap(r['date'], r[col]), axis=1)

print("\n✅ Former-name remapping applied to results, goalscorers, and shootouts")

# ---- Step 4: Post-2000 filter ----
results_2000 = results_new[results_new['date'].dt.year >= 2000].copy()

# ---- Step 5: Tournament keep-list filter (established earlier) ----
tournaments_to_keep = [
    'FIFA World Cup', 'FIFA World Cup qualification',
    'UEFA Euro', 'UEFA Euro qualification', 'UEFA Nations League',
    'African Cup of Nations', 'African Cup of Nations qualification',
    'AFC Asian Cup', 'AFC Asian Cup qualification',
    'Copa América', 'Gold Cup',
    'CONCACAF Nations League', 'CONCACAF Nations League qualification',
    'Confederations Cup', 'Oceania Nations Cup',
    'WAFF Championship', 'Arab Cup', 'CAFA Nations Cup', 'Amílcar Cabral Cup'
]

clean_df = results_2000[results_2000['tournament'].isin(tournaments_to_keep)].copy()
clean_df = clean_df.dropna(subset=['home_score','away_score']).reset_index(drop=True)

print(f"\nFinal clean_matches shape : {clean_df.shape}")
print(f"Tournaments kept          : {clean_df['tournament'].nunique()}")
print(f"Date range                : {clean_df['date'].min()} to {clean_df['date'].max()}")

# ---- Save everything ----
clean_df.to_csv('../data/processed/clean_matches.csv', index=False)
upcoming_fixtures.to_csv('../data/wc2026/upcoming_fixtures.csv', index=False)
goals_df.to_csv('../data/raw/goalscorers.csv', index=False)
shootouts_df.to_csv('../data/raw/shootouts.csv', index=False)

print("\n✅ Saved: clean_matches.csv, upcoming_fixtures.csv (overwritten goalscorers.csv/shootouts.csv with remapped names)")